<a href="https://colab.research.google.com/github/sathvik-007/playground-o1onoyz6/blob/master/Optimization_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimization Challange

In [ ]:
"""
=======================================================================
PYSPARK INTERVIEW CHALLENGE: CODE REVIEW & OPTIMIZATION
=======================================================================

BUSINESS CONTEXT:
A junior engineer wrote the following script to process daily sales
transactions. The `transactions` dataset is massive (500GB), while the
`stores` dataset is a tiny dimensional lookup table (5MB).

The script works locally on sample data, but fails or hangs when pushed
to our production cluster.

YOUR TASK:
Review the code below. Use the "HINT" comments to identify the logical
errors, syntax issues, and performance bottlenecks. Explain why each is
a problem, and then rewrite the script to be production-ready.
=======================================================================
"""

from pyspark.sql import SparkSession
# HINT 1: Are we missing any imports needed for the logic below?

def process_daily_sales():
    spark = SparkSession.builder.appName("DailySalesProcessing").getOrCreate()

    # Load data (Assume transactions is 500GB, stores is 5MB)
    transactions_df = spark.read.parquet("s3://data/transactions/")
    stores_df = spark.read.csv("s3://data/stores.csv", header=True)

    # HINT 2: Consider how PySpark DataFrames manage state. Will this work?
    transactions_df.drop("unnecessary_metadata_column")

    # HINT 3: Performance bottleneck. Think about how Spark shuffles data
    # across the cluster when joining a 500GB table with a 5MB table.
    joined_df = transactions_df.join(
        stores_df,
        transactions_df.store_id == stores_df.store_id,
        "inner"
    )

    # HINT 4:
    def categorize_sale(amount):
        if amount > 1000:
            return 'High_Value'
        else:
            return 'Standard'

    from pyspark.sql.functions import udf
    from pyspark.sql.types import StringType
    categorize_udf = udf(categorize_sale, StringType())

    enriched_df = joined_df.withColumn("sale_category", categorize_udf("transaction_amount"))

    # HINT 5: Syntax/Reference error. Will this line execute successfully?
    filtered_df = enriched_df.filter(col("status") == "COMPLETED")

    # HINT 6:
    completed_records = filtered_df.collect()

    for row in completed_records:
        print(f"Processed transaction: {row['transaction_id']}")

if __name__ == "__main__":
    process_daily_sales()

# SQL Task 1

In [ ]:
%sql
/*
=======================================================================
SQL INTERVIEW CHALLENGE: CUSTOMER SALES ANALYSIS
=======================================================================

BUSINESS CONTEXT:
You are tasked with analyzing sales data for an international e-commerce
platform. You need to identify high-value customers both globally and
regionally to help the marketing team distribute VIP rewards.

TABLE SCHEMAS:
1. `customers`
   - customer_id (INT): Primary Key
   - customer_name (VARCHAR): Full name of the customer
   - country (VARCHAR): Customer''s country of residence

2. `orders`
   - order_id (INT): Primary Key
   - customer_id (INT): Foreign Key referencing customers.customer_id
   - order_date (DATE): The date the order was placed
   - order_amount (DECIMAL): The monetary value of the order

=======================================================================
YOUR TASKS:

Task 1: Global Top Performers
Write a query to identify the top 3 customers overall based on their
lifetime total order amount.
Expected Output: customer_name, total_order_amount

=======================================================================
*/

-- 1. Create Tables
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100),
    country VARCHAR(50)
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    order_amount DECIMAL(10, 2)
);

-- 2. Insert Sample Customers
INSERT INTO customers (customer_id, customer_name, country) VALUES
(1, 'Alice Smith', 'USA'),
(2, 'Bob Johnson', 'USA'),
(3, 'Charlie Brown', 'USA'),
(4, 'Diana Prince', 'USA'),
(5, 'Ethan Hunt', 'UK'),
(6, 'Fiona Gallagher', 'UK'),
(7, 'George Lucas', 'UK'),
(8, 'Hannah Abbott', 'UK'),
(9, 'Ian Malcolm', 'Canada'),
(10, 'Julia Roberts', 'Canada');

-- 3. Insert Sample Orders
-- USA Customers (Alice: 900, Bob: 300, Charlie: 450, Diana: 150)
INSERT INTO orders (order_id, customer_id, order_date, order_amount) VALUES
(101, 1, '2023-01-15', 400.00),
(102, 1, '2023-02-20', 500.00),
(103, 2, '2023-01-18', 300.00),
(104, 3, '2023-03-10', 450.00),
(105, 4, '2023-04-05', 150.00);

-- UK Customers (Ethan: 1200, Fiona: 800, George: 600, Hannah: 200)
INSERT INTO orders (order_id, customer_id, order_date, order_amount) VALUES
(106, 5, '2023-01-22', 600.00),
(107, 5, '2023-05-11', 600.00),
(108, 6, '2023-02-14', 800.00),
(109, 7, '2023-06-30', 600.00),
(110, 8, '2023-07-12', 200.00);

-- Canada Customers (Ian: 1500, Julia: 100)
INSERT INTO orders (order_id, customer_id, order_date, order_amount) VALUES
(111, 9, '2023-08-21', 1000.00),
(112, 9, '2023-09-15', 500.00),
(113, 10, '2023-10-01', 100.00);



SELECT
customer_name, SUM(order_amount) over(partition by ) total_order_amount

FROM
customers INNER JOIN
orders
ON customers.customer_id = orders.customer_id

In [ ]:
%sql
/*
=======================================================================
SQL INTERVIEW CHALLENGE: PERFORMANCE TUNING
=======================================================================

BUSINESS CONTEXT:
A junior developer wrote the following query to solve Task 1 (finding
the top 3 customers globally). It works perfectly in our dev environment
with 100 rows. However, in production—where the `orders` table has
50 million rows and the `customers` table has 5 million rows—this query
is timing out and bringing down the database.

YOUR TASK:
1. Identify the performance bottlenecks in the query below.
2. Rewrite the query to be highly optimized for large datasets.

=======================================================================
*/

SELECT
    c.customer_name,
    (SELECT SUM(order_amount) FROM orders o WHERE o.customer_id = c.customer_id) AS total_order_amount
FROM
    customers c
WHERE
    (SELECT SUM(order_amount) FROM orders o WHERE o.customer_id = c.customer_id) IS NOT NULL
ORDER BY
    (SELECT SUM(order_amount) FROM orders o WHERE o.customer_id = c.customer_id) DESC
LIMIT 3;

In [ ]:
%sql



# SQL Task 2

In [ ]:
%sql
/*
=======================================================================
SQL INTERVIEW CHALLENGE: DATA CLEANSING & DEDUPLICATION
=======================================================================

BUSINESS CONTEXT:
Due to a temporary glitch in our web registration API, our system
accidentally recorded duplicate entries for several newsletter sign-ups.
You need to analyze the extent of the issue and clean the database by
removing the duplicates, ensuring we only retain the original records.

TABLE SCHEMA:
`subscribers`
   - id (INT): Primary Key
   - email (VARCHAR): The subscriber's email address
   - signup_date (DATE): The date the record was created
   - status (VARCHAR): The account status (e.g., 'Active', 'Pending')

=======================================================================
YOUR TASKS:

Task 1: Identify the Duplicates
Write a SELECT query to find all email addresses that appear more than
once in the table. Your result should include the email address and
the number of times it appears.
Expected Output: email, duplicate_count

Task 2: Cleanse the Data
Write a query to DELETE the duplicate records from the table. For each
set of duplicate emails, you must KEEP the record with the lowest `id`
(which represents the original sign-up) and delete all subsequent copies.

Note: If you are using an environment where DELETE is restricted, you
may write a SELECT query that returns the fully cleansed dataset instead.
=======================================================================
*/

-- 1. Create Table
CREATE TABLE subscribers (
    id INT PRIMARY KEY,
    email VARCHAR(100),
    signup_date DATE,
    status VARCHAR(50)
);

-- 2. Insert Sample Data
INSERT INTO subscribers (id, email, signup_date, status) VALUES
-- Normal records (No duplicates)
(1, 'sarah.connor@example.com', '2023-10-01', 'Active'),
(2, 'john.wick@example.com', '2023-10-02', 'Active'),

-- Duplicate pair (Keep ID 3, drop ID 5)
(3, 'bruce.wayne@example.com', '2023-10-03', 'Active'),
(4, 'clark.kent@example.com', '2023-10-03', 'Pending'),
(5, 'bruce.wayne@example.com', '2023-10-03', 'Active'),

-- Triplicate entries (Keep ID 6, drop ID 8 and 9)
(6, 'peter.parker@example.com', '2023-10-04', 'Active'),
(7, 'tony.stark@example.com', '2023-10-05', 'Active'),
(8, 'peter.parker@example.com', '2023-10-04', 'Active'),
(9, 'peter.parker@example.com', '2023-10-06', 'Pending');

In [ ]:
%sql

select 1

# PySpark Task 1

In [ ]:
"""
=======================================================================
PYSPARK INTERVIEW CHALLENGE: SECURITY LOG ANALYSIS
=======================================================================

BUSINESS CONTEXT:
You are a Data Engineer on the Cyber Security team. We ingest streaming
JSON logs from our authentication servers. Your goal is to process these
logs through a Medallion Architecture (Bronze -> Silver) to detect
potential brute-force attacks and account takeovers.

INPUT SCHEMA (JSON):
{
  "timestamp": string (ISO 8601),
  "userId": string,
  "eventType": string (e.g., "login_attempt"),
  "details": {
    "success": boolean,
    "ipAddress": string
  }
}

=======================================================================
YOUR TASKS:

Task 1: The Bronze Layer (Ingestion & Cleansing)
Read the raw JSON data. Create a Bronze DataFrame where:
- The `timestamp` string is explicitly cast to a Spark TimestampType.
- The nested `details` struct is flattened into two separate, top-level
  columns: `is_success` (boolean) and `ip_address` (string).
- The original `details` column is dropped.

Task 2: The Silver Layer (Brute-Force Detection)
Using the Bronze DataFrame, identify users who may be victims of a
brute-force attack.
- Requirement: Find users who have more than 5 failed login attempts
  within ANY tumbling 10-minute window.
- Expected Output: userId, window_start, window_end, failed_count

Task 3: The Silver Layer (Account Takeover Detection)
Using the Bronze DataFrame, identify users whose accounts may have
been successfully breached.
- Requirement: Find users who had a sequence of at least 3 consecutive
  failed login attempts immediately followed by a successful login.
- Expected Output: userId, breach_timestamp (the time of the success)
=======================================================================
"""

from pyspark.sql import SparkSession
import json

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("SecurityLogAnalysis") \
    .master("local[*]") \
    .getOrCreate()

# 2. Sample Data Payload
# User-1: Normal user (1 fail, 1 success)
# User-2: Brute force victim (6 fails in 10 mins) -> Should trigger Task 2
# User-3: Account takeover (3 fails, then 1 success) -> Should trigger Task 3
raw_json_data = [
    # User 1 - Normal Activity
    '{"timestamp": "2025-06-21T10:01:00Z", "userId": "user-1", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "192.168.1.10"}}',
    '{"timestamp": "2025-06-21T10:02:00Z", "userId": "user-1", "eventType": "login_attempt", "details": {"success": true, "ipAddress": "192.168.1.10"}}',

    # User 2 - Brute Force (6 failures within 10:00 - 10:10 window)
    '{"timestamp": "2025-06-21T10:01:30Z", "userId": "user-2", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "10.0.0.5"}}',
    '{"timestamp": "2025-06-21T10:03:00Z", "userId": "user-2", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "10.0.0.5"}}',
    '{"timestamp": "2025-06-21T10:04:15Z", "userId": "user-2", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "10.0.0.5"}}',
    '{"timestamp": "2025-06-21T10:05:45Z", "userId": "user-2", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "10.0.0.5"}}',
    '{"timestamp": "2025-06-21T10:08:00Z", "userId": "user-2", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "10.0.0.5"}}',
    '{"timestamp": "2025-06-21T10:09:30Z", "userId": "user-2", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "10.0.0.5"}}',

    # User 3 - Account Takeover (3 fails, followed by a success)
    '{"timestamp": "2025-06-21T11:20:00Z", "userId": "user-3", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "172.16.0.8"}}',
    '{"timestamp": "2025-06-21T11:21:00Z", "userId": "user-3", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "172.16.0.8"}}',
    '{"timestamp": "2025-06-21T11:22:00Z", "userId": "user-3", "eventType": "login_attempt", "details": {"success": false, "ipAddress": "172.16.0.8"}}',
    '{"timestamp": "2025-06-21T11:25:00Z", "userId": "user-3", "eventType": "login_attempt", "details": {"success": true, "ipAddress": "172.16.0.8"}}'
]

# 3. Create initial RDD/DataFrame for the candidate to start with
rdd = spark.sparkContext.parallelize(raw_json_data)
raw_df = spark.read.json(rdd)

raw_df.show(truncate=False)

# --- CANDIDATE CODE GOES BELOW ---